# Portfolio - Optimization & Allocation

Notebook scaffold for portfolio optimization and allocation analysis.

In [ ]:
# Code Block 1: Notebook description

# Focused scaffold for optimization inputs only.

# Outputs from this notebook: invested tickers and directional signage.


In [ ]:
#paramters 
period = "max"
interval = "1d"

In [ ]:
# Code Block 2: Load Libraries

# Load Libraries

import numpy as np

import pandas as pd



import statsmodels

import statsmodels.api as sm

from statsmodels.tsa.stattools import coint

from IPython.display import display

import schwab

from schwab.auth import easy_client

from schwab.client import Client

import re 

import os

import matplotlib.pyplot as plt

import plotly.express as px

from plotly.subplots import make_subplots

import plotly.graph_objects as go

import sys

from pathlib import Path



def find_project_root(start: Path) -> Path:

    for candidate in (start, *start.parents):

        if (candidate / "pyproject.toml").exists() and (candidate / "Quantapp").is_dir():

            return candidate

    raise RuntimeError("Could not locate the project root containing the Quantapp package.")



PROJECT_ROOT = find_project_root(Path.cwd().resolve())

if str(PROJECT_ROOT) not in sys.path:

    sys.path.insert(0, str(PROJECT_ROOT))



import yfinance as yf

#from Quantapp.analytics import TimeSeriesAnalytics as Rolling

from Quantapp.data import MacroDataClient

from Quantapp.secrets import load_project_env, require_secret



load_project_env()



#qc = Rolling()

qe = MacroDataClient()

In [ ]:
# Code Block 3: Direction/sign helpers



OPTION_PATTERN = re.compile(

    r'^(?P<underlying>[A-Z]{1,6})(?P<expiration>\\d{6})(?P<option_type>[CP])(?P<strike>\\d{8})$'

)



def normalize_yf_ticker(ticker):

    if not isinstance(ticker, str):

        return ticker

    return ticker.strip().replace('/', '-')



def build_yf_ticker_map(tickers):

    return {ticker: normalize_yf_ticker(ticker) for ticker in tickers}



def infer_directional_signage(positions):

    option_rows = []



    for position in positions:

        instrument = position.get("instrument", {})

        if instrument.get("assetType") != "OPTION":

            continue



        raw_symbol = "".join((instrument.get("symbol", "") or "").split())

        match = OPTION_PATTERN.match(raw_symbol)

        if not match:

            continue



        option_rows.append(

            {

                "underlying": match.group("underlying"),

                "option_type": match.group("option_type"),

                "long_quantity": position.get("longQuantity", 0.0),

                "short_quantity": position.get("shortQuantity", 0.0),

                "average_price": position.get("averagePrice", 0.0),

            }

        )



    positions_df = pd.DataFrame(option_rows)

    if positions_df.empty:

        return positions_df, pd.Series(dtype=object), pd.Series(dtype=float)



    direction_factor = positions_df["option_type"].map({"C": 1.0, "P": -1.0}).fillna(0.0)

    positions_df["net_quantity"] = positions_df["long_quantity"] - positions_df["short_quantity"]

    positions_df["directional_value"] = positions_df["net_quantity"] * positions_df["average_price"] * direction_factor * 100



    net_direction = positions_df.groupby("underlying")["directional_value"].sum()

    sentiment_map = net_direction.apply(lambda x: "bullish" if x > 0 else ("bearish" if x < 0 else "neutral"))

    directional_signage = sentiment_map.map({"bullish": 1, "bearish": -1, "neutral": 0}).astype(int)



    return positions_df, sentiment_map, directional_signage


In [ ]:
# Code Block 4: Define Schwab parameters



CLIENT_ID = require_secret("SCHWAB_CLIENT_ID")

APP_SECRET = require_secret("SCHWAB_APP_SECRET")

CALLBACK_URL = os.getenv("SCHWAB_CALLBACK_URL", "https://127.0.0.1:8182")

TOKEN_PATH = os.getenv("SCHWAB_TOKEN_PATH", "schwab_token.json")


In [ ]:
# Code Block 5: Login to Schwab client
#Login to client (must be done manually, you will be prompted for the url)
from schwab.auth import client_from_manual_flow
from schwab.client import Client
import json
client = client_from_manual_flow(
    api_key=CLIENT_ID,
    app_secret=APP_SECRET,
    callback_url=CALLBACK_URL,
    token_path=TOKEN_PATH
)


In [ ]:
# Code Block 6: Retrieve account and market data
#retreive  account and market data

#account information: the account infromation of course
#positions_df:        the accounts current options positions
#invested_sybmols:     all of the symbols im currently invested in
#total margin:        the total amount of money under my control for each position
#benchmark_close      the historical closing prices of the benchmark data
#portfolio_close      the historical closing prices of the portfolio
#net_direction        the "net direction" of the options portfolio per position


#general account information
account_information = client.get_accounts().json()

#retrieve options options
#-------------------------------------------------------------------------------
acct_map = client.get_account_numbers().json()
if isinstance(acct_map, dict):
    accounts = acct_map.get("accounts") or acct_map.get("accountNumbers") or []
elif isinstance(acct_map, list):
    accounts = acct_map
else:
    accounts = []

if not accounts:
    raise ValueError("No Schwab accounts returned from get_account_numbers().")

first_account = accounts[0]
acct_hash = first_account.get('hashValue') if isinstance(first_account, dict) else None
if not acct_hash:
    raise ValueError("Missing hashValue in Schwab account numbers response.")

resp = client.get_account(acct_hash, fields=Client.Account.Fields.POSITIONS)
acct = resp.json()
positions = acct.get("securitiesAccount", {}).get("positions", [])
#-------------------------------------------------------------------------------

#parses positions for all options positions and formats into a simple dataframe
#-------------------------------------------------------------------------------
positions_df = pd.DataFrame()
option_metadata_rows = []
option_pattern = re.compile(r'^(?P<underlying>[A-Z]{1,6})(?P<expiration>\d{6})(?P<option_type>[CP])(?P<strike>\d{8})$')
today_normalized = pd.Timestamp.today().normalize()

for position in positions:
    position_symbol = position['instrument'].get('symbol')
    position_symbol = "".join(position_symbol.split())

    if position.get('instrument', {}).get('assetType') == 'OPTION':
        match = option_pattern.match(position_symbol)
        if match:
            expiration_code = match.group('expiration')
            expiration_date = pd.to_datetime('20' + expiration_code, format='%Y%m%d', errors='coerce')
            days_to_expiration = int((expiration_date.normalize() - today_normalized).days) if pd.notna(expiration_date) else pd.NA
            strike_price = int(match.group('strike')) / 1000
            option_metadata_rows.append(
                {
                    'symbol': position_symbol,
                    'underlying': match.group('underlying'),
                    'expiration': expiration_date,
                    'days_to_expiration': days_to_expiration,
                    'option_type': match.group('option_type'),
                    'strike': strike_price,
                    'underlying_symbol': position.get('instrument', {}).get('underlyingSymbol'),
                    'put_call': position.get('instrument', {}).get('putCall'),
                    'long_quantity': position.get('longQuantity', 0.0),
                    'short_quantity': position.get('shortQuantity', 0.0),
                    'net_quantity': position.get('longQuantity', 0.0) - position.get('shortQuantity', 0.0),
                    'average_price': position.get('averagePrice', 0.0),
                }
            )

positions_df = pd.DataFrame(option_metadata_rows)


#calculates informally the directionality of my portfolio
#---------------------------------------------------------------------------------------------------------
if not positions_df.empty:
    direction_factor = positions_df['option_type'].map({'C': 1, 'P': -1}).fillna(0)
    positions_df['directional_value'] = positions_df['net_quantity'] * positions_df['average_price'] * direction_factor * 100
    net_direction = positions_df.groupby('underlying')['directional_value'].sum()
    sentiment_map = net_direction.apply(lambda x: 'bullish' if x > 0 else ('bearish' if x < 0 else 'neutral'))
    option_sentiment = sentiment_map.to_dict()
    net_direction = net_direction.to_frame('directional_value').join(sentiment_map.rename('sentiment'))
    #display(net_direction.to_frame('directional_value').join(sentiment_map.rename('sentiment')))
else:
    option_sentiment = {}
    net_direction = pd.DataFrame(columns=['directional_value', 'sentiment'])
    
net_direction['sign'] = net_direction['sentiment'].map({'bullish': 1, 'bearish': -1, 'neutral': 0})
#--------------------------------------------------------------------------------------------------------



#retrieve invested symbols
#---------------------------------------------------------------------------------
organized_positions = {}
for pos in positions:
    symbol = pos.get("instrument", {}).get("underlyingSymbol", "")
    if symbol:
        organized_positions.setdefault(symbol, []).append(pos)
        
invested_symbols = list(organized_positions.keys())
#---------------------------------------------------------------------------------


#for each symbol in organized_positions, get the total invested amount
net_invested_amounts = {}
total_margin = {}

for symbol in invested_symbols:
    positions_list = organized_positions[symbol]
    net_sum = 0
    #print(positions_list[0].keys())
    
    for position in positions_list:
        averagePrice = position.get("averagePrice", 0)
        maintenance_requirement = position.get("maintenanceRequirement", 0) / 100
        putCall = position.get("instrument", {}).get("putCall", "N/A")
        short_quantity = position.get("shortQuantity", 0)
        long_quantity = position.get("longQuantity", 0)
        # If it's a PUT, treat price as negative
        if putCall == "PUT":
            averagePrice = -averagePrice

        # Calculate net invested amount using both long and short quantities
        net_invested = (long_quantity - short_quantity) * averagePrice * 100  # times 100 for options
        maintenance_requirement = maintenance_requirement * 100  # times 100 for options
        total_margin[symbol] = total_margin.get(symbol, 0) + maintenance_requirement

        # Print symbol, putCall, averagePrice, short_quantity, long_quantity, and net_invested
        #print(f"Symbol: {symbol}, Put/Call: {putCall}, Average Price: {averagePrice}, Short Quantity: {short_quantity}, Long Quantity: {long_quantity}, Net Invested: {net_invested}")

        net_sum += net_invested
    net_invested_amounts[symbol] = net_sum
    #display(f"Symbol: {symbol}, Net Invested Amount: {net_sum}")
    #display(f"Symbol: {symbol}, Total Margin Requirement: {total_margin[symbol]}")
#-------------------------------------------------------------------------------------

#retrieve core data for benchmark and portfolio
#--------------------------------------------------------------------------
benchmark_data = yf.Ticker('SPY').history(period=period, interval=interval)
invested_symbol_map = build_yf_ticker_map(invested_symbols)
portfolio_data = yf.download(
            tickers=list(invested_symbol_map.values()),
            period=period,
            interval=interval,
            auto_adjust=True,
            threads=True,
            progress=False
        )

portfolio_closing_prices = portfolio_data['Close']
if isinstance(portfolio_closing_prices, pd.Series):
    portfolio_closing_prices = portfolio_closing_prices.to_frame(name=invested_symbols[0])
else:
    portfolio_closing_prices = portfolio_closing_prices.rename(columns={yf_ticker: ticker for ticker, yf_ticker in invested_symbol_map.items()})
benchmark_close          = benchmark_data['Close']

#portfolioc_closing prices index is formated like 'YYYY-MM-DD' but benchmark_close index is formated like 'YYYY-MM-DD HH:MM:SS-00:00' 
#need to convert portfolio_closing_prices index to match benchmark_close index

#display(portfolio_closing_prices.index)
portfolio_closing_prices.index = portfolio_closing_prices.index.tz_localize(None)
benchmark_close.index = benchmark_close.index.tz_localize(None)
#--------------------------------------------------------------------------

invested_tickers = invested_symbols
directional_signage = net_direction['sign'] if 'sign' in net_direction else pd.Series(dtype=int)
directional_signage_table = directional_signage.rename('sign').reset_index().rename(columns={'underlying': 'ticker'})

display(directional_signage_table.sort_values('ticker').reset_index(drop=True))
raw_prices = portfolio_closing_prices.copy()

In [ ]:
# Mean-Variance Optimization (Max Sharpe) from raw_prices
import numpy as np
import pandas as pd
from scipy.optimize import minimize
import matplotlib.pyplot as plt

# 1) Build return series
returns = raw_prices.pct_change().dropna(how="all")
returns = returns.dropna(axis=1, how="all")  # remove dead columns

# Optional: align to invested_tickers if you want strict set
if "invested_tickers" in globals():
    cols = [c for c in invested_tickers if c in returns.columns]
    returns = returns[cols]

# 2) Estimate mu and Sigma (annualized)
trading_days = 252
mu = returns.mean() * trading_days
Sigma = returns.cov() * trading_days

n = len(mu)
if n == 0:
    raise ValueError("No valid return series found for optimization.")

# Risk-free rate assumption (adjust as needed)
rf = 0.04

# 3) Max Sharpe objective
def neg_sharpe(w, mu, Sigma, rf):
    port_ret = w @ mu.values
    port_vol = np.sqrt(w @ Sigma.values @ w)
    if port_vol <= 0:
        return 1e6
    return -((port_ret - rf) / port_vol)

# Constraints
cons = [{"type": "eq", "fun": lambda w: np.sum(w) - 1.0}]
bounds = [(0.0, 1.0)] * n  # long-only
w0 = np.repeat(1.0 / n, n)

res = minimize(
    neg_sharpe,
    w0,
    args=(mu, Sigma, rf),
    method="SLSQP",
    bounds=bounds,
    constraints=cons,
)

if not res.success:
    raise RuntimeError(f"Optimization failed: {res.message}")

weights = pd.Series(res.x, index=mu.index, name="weight").sort_values(ascending=False)

# Diagnostics
opt_ret = float(weights.values @ mu.values)
opt_vol = float(np.sqrt(weights.values @ Sigma.values @ weights.values))
opt_sharpe = (opt_ret - rf) / opt_vol

display(weights.to_frame())
print(f"Expected Return: {opt_ret:.2%}")
print(f"Expected Vol:    {opt_vol:.2%}")
print(f"Expected Sharpe: {opt_sharpe:.3f}")

# 4) Plot optimized weights
plot_weights = weights[weights > 0].sort_values(ascending=True)

plt.figure(figsize=(10, max(4, 0.35 * len(plot_weights))))
plt.barh(plot_weights.index, plot_weights.values)
plt.title("Optimized Portfolio Weights (Max Sharpe)")
plt.xlabel("Weight")
plt.ylabel("Ticker")
plt.xlim(0, max(0.05, plot_weights.max() * 1.15))
plt.grid(axis="x", alpha=0.25)
plt.tight_layout()
plt.show()

# 5) Plot random portfolios and highlight optimizer solution
num_portfolios = 3000
rand_w = np.random.dirichlet(np.ones(n), size=num_portfolios)
rand_ret = rand_w @ mu.values
rand_vol = np.sqrt(np.einsum("ij,jk,ik->i", rand_w, Sigma.values, rand_w))
rand_sharpe = (rand_ret - rf) / rand_vol

plt.figure(figsize=(10, 6))
sc = plt.scatter(rand_vol, rand_ret, c=rand_sharpe, cmap="viridis", s=10, alpha=0.6)
plt.colorbar(sc, label="Sharpe")
plt.scatter([opt_vol], [opt_ret], color="red", s=120, marker="*", label="Optimal (Max Sharpe)")
plt.title("Mean-Variance Opportunity Set")
plt.xlabel("Annualized Volatility")
plt.ylabel("Annualized Return")
plt.legend()
plt.grid(alpha=0.2)
plt.tight_layout()
plt.show()